<a href="https://colab.research.google.com/github/mehmetbozdemir24/Magibu/blob/main/benchmark/olcum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q sentence-transformers

In [3]:
%%capture
# --- Hücre 1: Kütüphane Kurulumları ---
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [4]:
# --- EN ÖNEMLİ KISIM: UNSLOTH İLK SIRADA OLMALI ---
from unsloth import FastVisionModel

# Unsloth'tan SONRA diğerlerini içe aktarıyoruz
import torch
import time
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# 1. Modeli ve Tokenizer'ı Yüklüyoruz
print("Model yükleniyor...")
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

# Çıkarım moduna alıyoruz
FastVisionModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Model yükleniyor...
==((====))==  Unsloth 2026.7.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
            (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear4bit(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
   

In [5]:
# 2. Anlamsal Benzerlik Modelini Yüklüyoruz
print("Anlamsal benzerlik modeli yükleniyor...")
anlamsal_benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# --- TEST FONKSİYONLARI ---

def cevap_dogru_mu(dogru_cevap_index, verilen_cevap, secenekler):
    harfler = ['A', 'B', 'C', 'D', 'E']
    dogru_harf = harfler[dogru_cevap_index]
    verilen_cevap = str(verilen_cevap).upper().strip()

    if dogru_harf == verilen_cevap:
        return True
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", "."]:
        return dogru_harf == verilen_cevap[0]
    else:
        encoded_cevap = anlamsal_benzerlik_modeli.encode([verilen_cevap])
        encoded_secenekler = anlamsal_benzerlik_modeli.encode(secenekler)
        benzerlik_listesi = anlamsal_benzerlik_modeli.similarity(encoded_cevap, encoded_secenekler).tolist()[0]
        en_yuksek_benzerlik = max(benzerlik_listesi)
        en_yuksek_benzerlik_index = benzerlik_listesi.index(en_yuksek_benzerlik)
        return en_yuksek_benzerlik_index == dogru_cevap_index

def ilerleme_cubugu(guncel, toplam, cubuk_uzunlugu=40):
    ilerleme = guncel / toplam
    blok = int(cubuk_uzunlugu * ilerleme)
    cubuk = "#" * blok + "-" * (cubuk_uzunlugu - blok)
    return f"[{cubuk}] {ilerleme * 100:.2f}%"

def modelden_cevap_al(model, tokenizer, prompt):
    messages = [
        {"role": "user", "content": [{"type": "text", "text": prompt}]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs['input_ids'].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    return response

def modeli_test_et(model, tokenizer, model_ismi):
    print(f"\n--- {model_ismi} İÇİN TEST BAŞLIYOR ---")
    mmlu_veri = pd.read_parquet("hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet")

    baslama_zamani = time.time()
    dogru_cevap_sayisi = 0

    for i in range(len(mmlu_veri)):
        soru = mmlu_veri.iloc[i]['soru'] + "\n"
        harfler = ['A', 'B', 'C', 'D', 'E']
        secenekler = mmlu_veri.iloc[i]['secenekler']

        for j in range(len(secenekler)):
            soru += f"{harfler[j]}: {secenekler[j]}\n"

        prompt = "Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun doğru cevabı olduğunu yaz. Örneğin 'A' veya 'B' gibi. Lütfen herhangi bir açıklama yapma!\nSoru: " + soru

        cevap = modelden_cevap_al(model, tokenizer, prompt)
        sonuc = cevap_dogru_mu(mmlu_veri.iloc[i]['cevap'], cevap, secenekler)

        if sonuc:
            dogru_cevap_sayisi += 1

        soru_index = i + 1
        simdi = time.time()
        cubuk = ilerleme_cubugu(soru_index, len(mmlu_veri))
        print(f"\r{soru_index} soru çözüldü. Geçen süre: {round(simdi - baslama_zamani, 1)}s. Doğru cevap: {dogru_cevap_sayisi}. Başarı: {round(dogru_cevap_sayisi / soru_index, 4)} İlerleme: {cubuk}", end="")

    ortalama = round(dogru_cevap_sayisi / len(mmlu_veri), 4)
    bitis_zamani = time.time()

    print(f"\n\n🎉 {model_ismi} Testi Tamamlandı!")
    print(f"Başarı Oranı: % {ortalama * 100}")

    return mmlu_veri

Anlamsal benzerlik modeli yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# --- TESTİ BAŞLAT ---
ham_model_sonuclari = modeli_test_et(model, tokenizer, "Qwen3.5-4B-Ham")


--- Qwen3.5-4B-Ham İÇİN TEST BAŞLIYOR ---
6200 soru çözüldü. Geçen süre: 9431.2s. Doğru cevap: 1151. Başarı: 0.1856 İlerleme: [########################################] 100.00%

🎉 Qwen3.5-4B-Ham Testi Tamamlandı!
Başarı Oranı: % 18.56


In [7]:
# --- HÜCRE: İNCE AYARLI (FINE-TUNED) MODELİ YÜKLEME VE TEST ETME ---

import gc
import torch

# 1. Eski modeli bellekten temizleyelim (CUDA Out of Memory hatasını önlemek için)
print("Eski model bellekten temizleniyor...")
del model
torch.cuda.empty_cache()
gc.collect()

# 2. Eğittiğin modeli Hugging Face'den yüklüyoruz
# Not: Unsloth, adapter repo ID'sini verdiğinde base modeli ve senin LoRA ağırlıklarını otomatik birleştirir.
print("Eğitilmiş model (nypgd/qwen3.5-4b-recipe-lora) yükleniyor...")
model, tokenizer = FastVisionModel.from_pretrained(
    "nypgd/qwen3.5-4b-recipe-lora",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

# Çıkarım (Inference) moduna alıyoruz
FastVisionModel.for_inference(model)

# 3. İnce Ayarlı Model İçin Testi Başlat
ince_ayar_model_sonuclari = modeli_test_et(model, tokenizer, "Qwen3.5-4B-Fine-Tuned")

# --- İsteğe Bağlı: Sonuçları Karşılaştırma ---
print("\n--- ÖZET KARŞILAŞTIRMA ---")
ham_basari = ham_model_sonuclari['cevap'].eq(ham_model_sonuclari['tahmin']).mean() if 'tahmin' in ham_model_sonuclari else "Hesaplanmadı"
ince_ayar_basari = ince_ayar_model_sonuclari['cevap'].eq(ince_ayar_model_sonuclari['tahmin']).mean() if 'tahmin' in ince_ayar_model_sonuclari else "Hesaplanmadı"

print("Ham Model Benchmark Tamamlandı, detaylar yukarıdadır.")
print("Fine-Tuned Benchmark Tamamlandı, detaylar yukarıdadır.")

Eski model bellekten temizleniyor...
Eğitilmiş model (nypgd/qwen3.5-4b-recipe-lora) yükleniyor...
==((====))==  Unsloth 2026.7.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]


--- Qwen3.5-4B-Fine-Tuned İÇİN TEST BAŞLIYOR ---
6200 soru çözüldü. Geçen süre: 14473.1s. Doğru cevap: 1151. Başarı: 0.1856 İlerleme: [########################################] 100.00%

🎉 Qwen3.5-4B-Fine-Tuned Testi Tamamlandı!
Başarı Oranı: % 18.56

--- ÖZET KARŞILAŞTIRMA ---
Ham Model Benchmark Tamamlandı, detaylar yukarıdadır.
Fine-Tuned Benchmark Tamamlandı, detaylar yukarıdadır.
